# 2025.1.22 Experiments

We would like to show three things in this jupyter notebook. 

1. Steady flow with drag 
2. Add additional friction constant
3. Add another slip/displacement force. 

First lets import the relevant files and choose the proper directory. 

In [10]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Modify base path for depending on your file structure.
BASE_PATH = "/Users/paxton/git"

# Specify path where .pkl files are located
target_dir = f"{BASE_PATH}/quail_volcano/scenarios/simple_1D_test"
# Specify path for Quail source code
source_dir = f"{BASE_PATH}/quail_volcano/src"
# Change to working directory
os.chdir(target_dir)


# Import quail modules
os.chdir(source_dir)

import meshing.tools as mesh_tools

import numerics.helpers.helpers as helpers
import numerics.timestepping.tools as stepper_tools

import physics.zerodimensional.zerodimensional as zerod
import physics.euler.euler as euler
import physics.navierstokes.navierstokes as navierstokes
import physics.scalar.scalar as scalar
import physics.chemistry.chemistry as chemistry
import physics.multiphasevpT.multiphasevpT as multiphasevpT

import processing.readwritedatafiles as readwritedatafiles
import processing.post as post
import processing.plot as plot

import solver.DG as DG
import solver.ADERDG as ADERDG
import solver.tools as solver_tools

import time
import multiprocessing as mp  
from multidomain import Domain, Observer

os.chdir(target_dir)

In [11]:
from IPython.display import HTML
from matplotlib import animation

def animate_conduit_pressure(folder, iterations=100):
    fig = plt.figure(figsize=(8,6.4))
    ax = fig.add_subplot(111,autoscale_on=False,\
                            xlim=(0,-1000),ylim=(0,10))

    density_line,  = ax.plot([], [], color="blue")
    ax.set_xlabel("Depth (m)")
    ax.set_ylabel("Pressure (MPa)")

    time_template = 'time = %.2f [s]'
    time_text = ax.text(0.7,0.9,'',transform=ax.transAxes)

    def init():
        density_line.set_data([], [])
        time_text.set_text("")
        return density_line, time_text

    def animate(i):
        solver = readwritedatafiles.read_data_file(f"{folder}/test_output_{i}.pkl")
        flag_non_physical = True
        p = solver.physics.compute_additional_variable("Pressure", solver.state_coeffs, flag_non_physical)

        # Get the position of of each nodal points (location corresponding to each entry of pDensityX)
        nodal_pts = solver.basis.get_nodes(solver.order)
        # Allocate [ne] x [nb, ndims]
        x = np.empty((solver.mesh.num_elems,) + nodal_pts.shape)
        for elem_ID in range(solver.mesh.num_elems):
            # Fill coordinates in physical space
            x[elem_ID] = mesh_tools.ref_to_phys(solver.mesh, elem_ID, nodal_pts)
        
        density_line.set_data(x.ravel(), p.ravel()/1e6)
        time_text.set_text(time_template % solver.time)

        return density_line, time_text

    plt.close()
    return animation.FuncAnimation(fig, animate, np.arange(iterations), blit=False, init_func=init, interval=50)

## 1. Depressurizing tube

- No source term
- Compressible fluids (Note that as a result we need to specify all the information _but_ pressure for the inflow, and for the outflow we only need to reference pressure)
- BC:
    - SlipWall on one side.
    - PressureOutlet1D on the other side. 
- IC 
    - No initial velocity. 
    - Steady state pressure

In [16]:
HTML(animate_conduit_pressure("depressurizing_tube").to_html5_video())

## 1.1 Add friction 

We set a shear friction term which is related to the radius of the conduit, in the below example we set the friction term to correspond with a conduit of radius $50m$. 

In [15]:
HTML(animate_conduit_pressure("depressurizing_tube_with_friction").to_html5_video())

## 2. Steady state flow

- No water, no air, no crystals 
- constant viscosity. 
- g = 0 


In [23]:
# Still in the works 
HTML(animate_conduit_pressure("steady_state_flow").to_html5_video())